# 1. Post-hoc robustness and explainability boundary
This study is restricted to the 1,434-row development set. The Notebook-05 Random Forest remains the official locked Stage-7 model. These post-hoc experiments explain its predictions, test coordinate removal and a safe category-aware SMOTENC design, and compare development-only robustness. They cannot replace the official model or evaluate the previously observed holdout. A stronger candidate would require separate formal reassessment and new external evaluation.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT","4")
from pathlib import Path
import json,hashlib,time,warnings,platform
from functools import partial
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.model_selection import StratifiedKFold,RepeatedStratifiedKFold,LeaveOneGroupOut,learning_curve
from sklearn.feature_selection import SelectKBest,mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score,average_precision_score,balanced_accuracy_score,
 confusion_matrix,f1_score,log_loss,matthews_corrcoef,precision_recall_fscore_support,
 precision_score,recall_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC
import sklearn,imblearn
ROOT=Path("..");RESULTS=ROOT/"reports/posthoc_robustness";RESULTS.mkdir(parents=True,exist_ok=True)
CLASS_ORDER=["Functional","Partially functional","Abandoned or not functional"]
CLASS_TO_ID={v:i for i,v in enumerate(CLASS_ORDER)};CLASS_IDS=np.arange(3);RANDOM_STATE=42
def sha256(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
LOCK_PATH=ROOT/"reports/model_optimization/final_selection_locked.json"
NB05_PATH=Path("05_model_optimization.ipynb")
LOCK_HASH_BEFORE=sha256(LOCK_PATH);NB05_HASH_BEFORE=sha256(NB05_PATH)
READ_PATHS=[]
print({"python":platform.python_version(),"sklearn":sklearn.__version__,"imblearn":imblearn.__version__})

# 2. Official locked model specification
Load exact parameters and feature policy from the Notebook-05 lock. The reference copy in this study is read-only evidence; the original lock and Notebook 05 must retain their starting hashes. Stage-7 development results provide a consistency target, not a source of new hyperparameters.

In [ ]:
LOCK=json.loads(LOCK_PATH.read_text(encoding="utf-8"))
assert LOCK["model_family"]=="Random Forest" and LOCK["preprocessing_variant"]=="tree_base"
assert LOCK["feature_set"]=="base" and LOCK["imbalance_strategy"]=="none"
assert LOCK["class_mapping"]==CLASS_TO_ID and LOCK["development_rows"]==1434
assert LOCK["feature_selection_k"]==60 and LOCK["rare_min_frequency"]==10
STAGE7=ROOT/"reports/model_optimization"
stage7_cv=pd.read_csv(STAGE7/"finalist_cv_summary.csv")
stage7_class=pd.read_csv(STAGE7/"finalist_oof_class_metrics.csv")
stage7_country=pd.read_csv(STAGE7/"finalist_country_robustness.csv")
stage7_compare=pd.read_csv(STAGE7/"baseline_vs_optimized.csv")
saved_reference=stage7_cv[stage7_cv.finalist==LOCK["finalist"]].iloc[0]
assert abs(saved_reference.macro_f1-LOCK["primary_cv_macro_f1"])<1e-12
(RESULTS/"reference_locked_model.json").write_text(json.dumps(LOCK,indent=2)+"\n",encoding="utf-8")
print("Official Stage-7 model remains locked:",LOCK["finalist"])
display(LOCK)

# 3. Development-only data
The only model data read here are X_train_raw.csv and y_train.csv. Raw predictor count, class map and row count are asserted. This notebook never reads the final assessment split or its metrics for a model decision.

In [ ]:
DATA=ROOT/"data/processed/modern"
def read_development(name):
 path=DATA/name;READ_PATHS.append(str(path.resolve()))
 return pd.read_csv(path)
X_train=read_development("X_train_raw.csv")
y_text=read_development("y_train.csv").iloc[:,0]
metadata=json.loads((DATA/"preprocessing_variants.json").read_text(encoding="utf-8"))
assert X_train.shape==(1434,32) and len(y_text)==1434
assert list(X_train.columns)==metadata["raw_predictor_columns"]
assert set(y_text)==set(CLASS_ORDER)
y=y_text.map(CLASS_TO_ID).astype(int)
print("Development shape:",X_train.shape,"class counts:",y_text.value_counts().to_dict())

# 4. Exact Notebook-03 preprocessing and locked RF reconstruction
The definitions below reproduce the current Notebook-03 implementation through Notebook 04. The reference pipeline uses tree_base, rare threshold 10, mutual-information SelectKBest(k=60), and the exact locked RF parameters. All fitted statistics and selected features are learned inside each training fold.

In [ ]:
class FixedDataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned = X.copy()
        cleaned["wc_savings_wp"] = cleaned["wc_savings_wp"].replace({999.0: np.nan, 888.0: np.nan})
        return cleaned

_clean_preview = FixedDataCleaner().fit_transform(X_train)
print("Training sentinel counts before:", X_train["wc_savings_wp"].isin([999, 888]).sum())
print("Training sentinel counts after:", _clean_preview["wc_savings_wp"].isin([999, 888]).sum())
assert X_train["wc_savings_wp"].equals(X_train.copy()["wc_savings_wp"])

class FeatureEngineer(BaseEstimator, TransformerMixin):
    # Adds engineered features. All fitted statistics come from the data passed
    # to .fit() only (i.e. the training split, when used inside the full pipeline).

    def fit(self, X, y=None):
        wp_age_train = X["wp_age"].astype(float)
        self.wp_age_median_ = wp_age_train.median()
        wp_age_filled = wp_age_train.fillna(self.wp_age_median_)
        q1, q2 = wp_age_filled.quantile([1 / 3, 2 / 3])
        self.wp_age_edges_ = (float(q1), float(q2))
        return self

    def transform(self, X):
        X = X.copy()

        # people_per_household: guard against division by zero / missing inputs
        qtyhh_safe = X["qtyhh_wp"].replace(0, np.nan)
        X["people_per_household"] = X["qtypeople_wp"] / qtyhh_safe

        # was_rehabilitated: clean 0/1/NaN mirror of rehabyn
        X["was_rehabilitated"] = X["rehabyn"].map({"Yes": 1, "No": 0})

        # has_management_committee: Water committee vs. everything else
        X["has_management_committee"] = (X["whomanage_wp"] == "Water committee").astype(int)

        # rehab_age: preserve structural meaning (fill 0 + explicit indicator)
        X["rehab_age_missing"] = X["rehab_age"].isna().astype(int)
        X["rehab_age"] = X["rehab_age"].fillna(0)

        # pop_1000: separate missing-indicator (kept apart from the log1p branch)
        X["pop_1000_missing"] = X["pop_1000"].isna().astype(int)

        # wp_age_group: bin using edges fitted on the training split only
        wp_age_filled = X["wp_age"].fillna(self.wp_age_median_)
        q1, q2 = self.wp_age_edges_
        X["wp_age_group"] = pd.cut(
            wp_age_filled,
            bins=[-np.inf, q1, q2, np.inf],
            labels=["New", "Established", "Old"],
        ).astype(str)

        return X


# Quick sanity preview, fit on the training split only
_fe_preview = FeatureEngineer().fit(X_train)
print(f"wp_age_group bin edges (fitted on X_train only): {_fe_preview.wp_age_edges_}")
_preview_out = _fe_preview.transform(X_train.head())
_preview_out[[
    "wp_age", "wp_age_group", "rehabyn", "was_rehabilitated",
    "whomanage_wp", "has_management_committee",
    "qtypeople_wp", "qtyhh_wp", "people_per_household",
    "rehab_age", "rehab_age_missing", "pop_1000", "pop_1000_missing",
]]

CAT_NOT_APPLICABLE_COLS = ["pumptype", "drillmethod", "piped_source", "piped_pump"]
CAT_UNKNOWN_RAW_COLS = [
    "country", "admin1", "wptype", "whomanage_wp", "rehabyn", "season",
    "cwfunded_wp", "wc_present_wp", "paytocollect_wp", "lockedfullday_wp",
]
ORDINAL_COLS = ["wc_admin_index_wp", "wc_finance_index_wp", "wc_mgmt_index_wp", "wc_maint_index_wp"]
ORDINAL_CATEGORY_ORDER = ["Inadequate", "Minimum", "Moderate", "Advanced", "Missing"]
RARE_MIN_FREQUENCY_CANDIDATES = [5, 10, 20]

def remap_ordinal_missing_code(codes):
    return np.where(codes == 4, -1, codes)

print("Structural categorical:", CAT_NOT_APPLICABLE_COLS)
print("Unknown categorical:", CAT_UNKNOWN_RAW_COLS)
print("Ordinal:", ORDINAL_COLS)

NUM_LOG_CANDIDATES = ["wp_age", "qtypeople_wp", "qtyhh_wp", "pop_1000"]
NUM_OTHER = ["elevation_wp", "latitude_wp", "longitude_wp", "rehab_age"]
NUM_WITH_INDICATOR = [
    "annual_rain", "wpqty_wp", "qtyhh_c_wp", "improved_wponly_wp",
    "wc_savings_wp", "pumpstrokes",
]
SEMANTIC_FLAGS = ["rehab_age_missing", "pop_1000_missing"]
OPTIONAL_NUM_WITH_INDICATOR = ["people_per_household", "was_rehabilitated"]
OPTIONAL_FLAGS = ["has_management_committee"]
OPTIONAL_CATEGORICAL = ["wp_age_group"]
print("Log candidates:", NUM_LOG_CANDIDATES)

RAW_PREDICTOR_COLS = (
    CAT_NOT_APPLICABLE_COLS + CAT_UNKNOWN_RAW_COLS + ORDINAL_COLS
    + NUM_LOG_CANDIDATES + NUM_OTHER + NUM_WITH_INDICATOR
)
assert len(RAW_PREDICTOR_COLS) == 32
assert len(set(RAW_PREDICTOR_COLS)) == 32
assert set(RAW_PREDICTOR_COLS) == set(X_train.columns)
print("Approved raw predictor count:", len(RAW_PREDICTOR_COLS))

class CategoricalStringifier(BaseEstimator, TransformerMixin):
    """Keep imputed nominal columns as CatBoost-safe Python strings."""

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        output = X.copy()
        for column in output.columns:
            output[column] = output[column].astype(str)
        return output

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)


def catboost_categorical_feature_names(include_optional_engineered_features):
    """Exact transformed DataFrame names, in ColumnTransformer order."""
    unknown_cols = CAT_UNKNOWN_RAW_COLS + (OPTIONAL_CATEGORICAL if include_optional_engineered_features else [])
    return ([f"cat_not_applicable__{column}" for column in CAT_NOT_APPLICABLE_COLS]
            + [f"cat_unknown__{column}" for column in unknown_cols])


def build_catboost_native_pipeline(include_optional_engineered_features):
    """Reuse the existing tree numeric/ordinal branches; replace only nominal OHE."""
    tree_reference = build_preprocessing_pipeline(
        "tree_boosting", include_optional_engineered_features,
        rare_min_frequency=10, use_log_transform=False, scale_numeric=False,
    )
    tree_branches = tree_reference.named_steps["preprocessing"].transformers
    native_branches = []
    for name, transformer, columns in tree_branches:
        if name in {"cat_not_applicable", "cat_unknown"}:
            fill = "Not applicable" if name == "cat_not_applicable" else "Unknown"
            transformer = Pipeline([
                ("impute", SimpleImputer(strategy="constant", fill_value=fill)),
                ("to_strings", CategoricalStringifier()),
            ])
        native_branches.append((name, transformer, columns))
    preprocessor = ColumnTransformer(
        native_branches, remainder="drop", verbose_feature_names_out=True,
    )
    preprocessor.set_output(transform="pandas")
    return Pipeline([
        ("fixed_cleaning", FixedDataCleaner()),
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", preprocessor),
    ])


def build_preprocessing_pipeline(model_family, include_optional_engineered_features,
                                 rare_min_frequency=10, use_log_transform=None,
                                 scale_numeric=None):
    if model_family == "catboost_native":
        # The existing builder default is 10 for OHE families. It is ignored here;
        # explicit alternative rare thresholds are invalid for native categories.
        if rare_min_frequency not in (None, 10):
            raise ValueError("rare_min_frequency is not applicable to catboost_native")
        if use_log_transform not in (None, False) or scale_numeric not in (None, False):
            raise ValueError("CatBoost-native baseline does not use log transforms or scaling")
        return build_catboost_native_pipeline(include_optional_engineered_features)
    if model_family not in {"scale_sensitive", "tree_boosting"}:
        raise ValueError("model_family must be scale_sensitive, tree_boosting, or catboost_native")
    if not isinstance(rare_min_frequency, int) or rare_min_frequency < 1:
        raise ValueError("rare_min_frequency must be a positive integer")
    if use_log_transform is None:
        use_log_transform = model_family == "scale_sensitive"
    if scale_numeric is None:
        scale_numeric = model_family == "scale_sensitive"

    def categorical_branch(fill_value):
        return Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value=fill_value)),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                      min_frequency=rare_min_frequency, sparse_output=False)),
        ])

    ordinal_branch = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("ordinal", OrdinalEncoder(categories=[ORDINAL_CATEGORY_ORDER] * len(ORDINAL_COLS))),
        ("remap_missing", FunctionTransformer(remap_ordinal_missing_code,
                                                 feature_names_out="one-to-one")),
    ])
    log_steps = [("impute", SimpleImputer(strategy="median"))]
    if use_log_transform:
        log_steps.append(("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")))
    if scale_numeric:
        log_steps.append(("scale", StandardScaler()))
    other_steps = [("impute", SimpleImputer(strategy="median"))]
    indicator_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale_numeric:
        other_steps.append(("scale", StandardScaler()))
        indicator_steps.append(("scale", StandardScaler()))

    unknown_cols = CAT_UNKNOWN_RAW_COLS + (OPTIONAL_CATEGORICAL if include_optional_engineered_features else [])
    indicator_cols = NUM_WITH_INDICATOR + (OPTIONAL_NUM_WITH_INDICATOR if include_optional_engineered_features else [])
    flags = SEMANTIC_FLAGS + (OPTIONAL_FLAGS if include_optional_engineered_features else [])
    preprocessor = ColumnTransformer([
        ("cat_not_applicable", categorical_branch("Not applicable"), CAT_NOT_APPLICABLE_COLS),
        ("cat_unknown", categorical_branch("Unknown"), unknown_cols),
        ("ordinal_wc_index", ordinal_branch, ORDINAL_COLS),
        ("num_skew", Pipeline(log_steps), NUM_LOG_CANDIDATES),
        ("num_other", Pipeline(other_steps), NUM_OTHER),
        ("num_indicator", Pipeline(indicator_steps), indicator_cols),
        ("semantic_flags", "passthrough", flags),
    ], remainder="drop", verbose_feature_names_out=True)
    preprocessor.set_output(transform="pandas")
    return Pipeline([
        ("fixed_cleaning", FixedDataCleaner()),
        ("feature_engineering", FeatureEngineer()),
        ("preprocessing", preprocessor),
    ])

VARIANTS = {
    "scaled_base": dict(model_family="scale_sensitive", include_optional_engineered_features=False,
                        rare_min_frequency=10, use_log_transform=True, scale_numeric=True),
    "scaled_engineered": dict(model_family="scale_sensitive", include_optional_engineered_features=True,
                              rare_min_frequency=10, use_log_transform=True, scale_numeric=True),
    "tree_base": dict(model_family="tree_boosting", include_optional_engineered_features=False,
                      rare_min_frequency=10, use_log_transform=False, scale_numeric=False),
    "tree_engineered": dict(model_family="tree_boosting", include_optional_engineered_features=True,
                            rare_min_frequency=10, use_log_transform=False, scale_numeric=False),
    "catboost_native_base": dict(model_family="catboost_native",
                                 include_optional_engineered_features=False,
                                 rare_min_frequency=None, use_log_transform=False,
                                 scale_numeric=False, one_hot_encode=False,
                                 native_categorical_handling=True,
                                 categorical_features=catboost_categorical_feature_names(False)),
    "catboost_native_engineered": dict(model_family="catboost_native",
                                       include_optional_engineered_features=True,
                                       rare_min_frequency=None, use_log_transform=False,
                                       scale_numeric=False, one_hot_encode=False,
                                       native_categorical_handling=True,
                                       categorical_features=catboost_categorical_feature_names(True)),
}
candidate_pipelines = {
    name: build_preprocessing_pipeline(**{
        key: value for key, value in config.items()
        if key not in {"one_hot_encode", "native_categorical_handling", "categorical_features"}
    })
    for name, config in VARIANTS.items()
}
print("Unfitted candidate variants:", list(candidate_pipelines))

In [ ]:
assert X_train.shape==(1434,32) and set(X_train.columns)==set(RAW_PREDICTOR_COLS)
tree_full=clone(candidate_pipelines["tree_base"]).fit_transform(X_train)
assert tree_full.shape==(1434,102) and np.isfinite(tree_full.to_numpy(dtype=float)).all()
assert metadata["variants"]["tree_base"]["rare_min_frequency"]==LOCK["rare_min_frequency"]
def locked_pipeline(remove=()):
 pre=build_preprocessing_pipeline("tree_boosting",False,
  rare_min_frequency=LOCK["rare_min_frequency"],use_log_transform=False,scale_numeric=False)
 if remove:
  ct=pre.named_steps["preprocessing"]
  ct.transformers=[(name,trans,[col for col in cols if col not in remove] if isinstance(cols,list) else cols)
   for name,trans,cols in ct.transformers]
 return Pipeline([("preprocessing",pre),
  ("selection",SelectKBest(partial(mutual_info_classif,random_state=42),k=LOCK["feature_selection_k"])),
  ("classifier",RandomForestClassifier(random_state=42,n_jobs=2,**LOCK["hyperparameters"]))])
assert clone(locked_pipeline()) is not None
print("Locked tree preprocessing parity PASS: 1434x102; selector k",LOCK["feature_selection_k"])

# 5. Common splits and metrics
All candidates use identical repeated stratified 5 by 3 splits for primary comparison and one fixed shuffled stratified five-fold OOF pass. Macro-F1 is primary; Balanced Accuracy, Macro Recall/Precision, MCC, Weighted F1, Accuracy, Log Loss, macro AP and Partial-class Precision/Recall/F1/AP are supporting metrics. Training scores, validation SD and train-CV gaps diagnose generalization.

In [ ]:
REPEATED=list(RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=42).split(X_train,y))
OOF=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(X_train,y))
assert len(REPEATED)==15 and len(OOF)==5
def aligned_proba(estimator,raw):
 out=np.zeros((len(raw),3),float)
 for j,k in enumerate(estimator.classes_):out[:,int(k)]=raw[:,j]
 sums=out.sum(axis=1,keepdims=True)
 assert np.isfinite(out).all() and (sums>0).all()
 return out/sums
def metrics(truth,pred,prob):
 p,r,f,_=precision_recall_fscore_support(truth,pred,labels=CLASS_IDS,zero_division=0)
 ap=np.array([average_precision_score((truth==k).astype(int),prob[:,k]) for k in CLASS_IDS])
 return {"macro_f1":f1_score(truth,pred,labels=CLASS_IDS,average="macro",zero_division=0),
  "balanced_accuracy":balanced_accuracy_score(truth,pred),
  "macro_recall":recall_score(truth,pred,labels=CLASS_IDS,average="macro",zero_division=0),
  "macro_precision":precision_score(truth,pred,labels=CLASS_IDS,average="macro",zero_division=0),
  "mcc":matthews_corrcoef(truth,pred),
  "weighted_f1":f1_score(truth,pred,labels=CLASS_IDS,average="weighted",zero_division=0),
  "accuracy":accuracy_score(truth,pred),"log_loss":log_loss(truth,prob,labels=CLASS_IDS),
  "macro_ap":float(ap.mean()),"partial_precision":float(p[1]),
  "partial_recall":float(r[1]),"partial_f1":float(f[1]),"partial_ap":float(ap[1])}
def evaluate_repeated(name,factory):
 rows=[]
 for n,(tr,va) in enumerate(REPEATED,1):
  pipe=factory();start=time.perf_counter();pipe.fit(X_train.iloc[tr],y.iloc[tr])
  train_pred=pipe.predict(X_train.iloc[tr]).astype(int).reshape(-1)
  pred=pipe.predict(X_train.iloc[va]).astype(int).reshape(-1)
  prob=aligned_proba(pipe.named_steps["classifier"],pipe.predict_proba(X_train.iloc[va]))
  val=metrics(y.iloc[va].to_numpy(),pred,prob)
  train_f1=f1_score(y.iloc[tr],train_pred,labels=CLASS_IDS,average="macro",zero_division=0)
  rows.append({"configuration":name,"repeat":(n-1)//5+1,"fold":(n-1)%5+1,
   "fit_seconds":time.perf_counter()-start,"train_macro_f1":train_f1,
   "train_cv_gap":train_f1-val["macro_f1"],**val})
 return pd.DataFrame(rows)
def evaluate_oof(name,factory):
 pred=np.full(len(y),-1,int);prob=np.full((len(y),3),np.nan);seen=np.zeros(len(y),int)
 for tr,va in OOF:
  pipe=factory();pipe.fit(X_train.iloc[tr],y.iloc[tr])
  pred[va]=pipe.predict(X_train.iloc[va]).astype(int).reshape(-1)
  prob[va]=aligned_proba(pipe.named_steps["classifier"],pipe.predict_proba(X_train.iloc[va]))
  seen[va]+=1
 assert np.all(seen==1) and np.isfinite(prob).all()
 return metrics(y.to_numpy(),pred,prob),pred,prob
def summarize(folds):
 numeric=[v for v in folds.columns if v not in ["configuration","repeat","fold"]]
 row={"configuration":folds.configuration.iloc[0]}
 for col in numeric:row[col]=folds[col].mean()
 row["sd_macro_f1"]=folds.macro_f1.std(ddof=1)
 return row

# 6. Locked RF reference on common folds
Rerun only the exact locked RF through the common evaluator; no Notebook-05 search is repeated. Its repeated-CV Macro-F1 should agree with the saved Stage-7 reference within ordinary deterministic tolerance. This direct run is the comparator for all post-hoc candidates.

In [ ]:
reference_folds=evaluate_repeated("Locked RF",locked_pipeline)
reference_folds.to_csv(RESULTS/"locked_rf_repeated_cv.csv",index=False)
reference_summary=pd.DataFrame([summarize(reference_folds)])
reference_summary.to_csv(RESULTS/"locked_rf_summary.csv",index=False)
assert abs(reference_summary.macro_f1.iloc[0]-LOCK["primary_cv_macro_f1"])<.01
reference_oof,reference_pred,reference_prob=evaluate_oof("Locked RF",locked_pipeline)
print("Locked reference repeated CV:");display(reference_summary.round(4))
print("Locked reference OOF Partial:",{k:round(reference_oof[k],4) for k in ["partial_precision","partial_recall","partial_f1","partial_ap"]})

# 7. Raw-feature permutation importance on validation folds
For each fixed five-fold split, fit the locked complete pipeline on the training fold, then shuffle one of the 32 raw validation columns five times. Importance is the decrease in validation Macro-F1. Aggregate fold means, SD and positive-fold counts. This measures model reliance under these validation splits; it does not establish causality.

In [ ]:
importance_rows=[]
for fold,(tr,va) in enumerate(OOF,1):
 pipe=locked_pipeline();pipe.fit(X_train.iloc[tr],y.iloc[tr])
 result=permutation_importance(pipe,X_train.iloc[va],y.iloc[va],
  scoring="f1_macro",n_repeats=5,random_state=42,n_jobs=1)
 baseline_fold=f1_score(y.iloc[va],pipe.predict(X_train.iloc[va]),average="macro")
 for j,col in enumerate(X_train.columns):
  importance_rows.append({"fold":fold,"raw_feature":col,"baseline_macro_f1":baseline_fold,
   "importance_mean":float(result.importances_mean[j]),
   "importance_sd_within_fold":float(result.importances_std[j])})
permutation_folds=pd.DataFrame(importance_rows)
raw_importance=(permutation_folds.groupby("raw_feature").agg(
 mean_importance=("importance_mean","mean"),
 sd_across_folds=("importance_mean","std"),
 positive_folds=("importance_mean",lambda x:int((x>0).sum())),
 mean_within_fold_sd=("importance_sd_within_fold","mean"))
 .sort_values("mean_importance",ascending=False).reset_index())
raw_importance.to_csv(RESULTS/"raw_feature_permutation_importance.csv",index=False)
display(raw_importance.head(15).round(4))

# 8. Selected processed-feature importance
Fit the unchanged locked pipeline on all development rows to map the exact selected 60 transformed names to the RF impurity importances. These values describe tree split use and can favor continuous or high-cardinality features. Validation-fold raw permutation importance is the stronger predictive explanation; neither method proves causality.

In [ ]:
full_locked=locked_pipeline();full_locked.fit(X_train,y)
names=full_locked.named_steps["preprocessing"].named_steps["preprocessing"].get_feature_names_out()
mask=full_locked.named_steps["selection"].get_support()
assert len(names)==len(mask) and mask.sum()==LOCK["feature_selection_k"]
selected_names=np.asarray(names)[mask]
imp=full_locked.named_steps["classifier"].feature_importances_
assert len(selected_names)==len(imp)==60 and np.isclose(imp.sum(),1)
processed_importance=(pd.DataFrame({"selected_processed_feature":selected_names,"rf_impurity_importance":imp})
 .sort_values("rf_impurity_importance",ascending=False).reset_index(drop=True))
processed_importance.to_csv(RESULTS/"selected_processed_feature_importance.csv",index=False)
display(processed_importance.head(20).round(4))

# 9. Coordinate ablation
Remove only latitude_wp and longitude_wp from the locked tree ColumnTransformer. The raw rows, cleaner, base feature policy, rare threshold, k=60 selector, RF parameters, no-imbalance strategy and common folds remain unchanged. No dummy coordinates are inserted and no parameters are retuned.

In [ ]:
coords=("latitude_wp","longitude_wp")
coord_factory=lambda:locked_pipeline(remove=coords)
coord_pre=coord_factory().named_steps["preprocessing"].named_steps["preprocessing"]
out=coord_pre.fit_transform(FeatureEngineer().fit_transform(FixedDataCleaner().fit_transform(X_train)))
assert out.shape[0]==1434 and out.shape[1]>=60
assert not any("latitude_wp" in n or "longitude_wp" in n for n in out.columns)
coordinate_folds=evaluate_repeated("RF without coordinates",coord_factory)
coordinate_folds.to_csv(RESULTS/"coordinate_ablation_folds.csv",index=False)
coordinate_summary=pd.DataFrame([summarize(coordinate_folds)])
coordinate_summary.to_csv(RESULTS/"coordinate_ablation_summary.csv",index=False)
coordinate_oof,coordinate_pred,coordinate_prob=evaluate_oof("RF without coordinates",coord_factory)
display(coordinate_summary.round(4))

# 10. Safe mixed representation for SMOTENC
A dedicated fold-fitted transformer keeps 14 nominal and 4 committee-index variables categorical, imputing them with Notebook-03 Not applicable, Unknown and Missing semantics. Sixteen numeric/flag columns receive training-fold median imputation; no NaN reaches SMOTENC. The post-sampling encoder one-hot encodes nominal columns with the same rare-category policy, ordinal-encodes the committee indices with the same Missing to -1 remap, and leaves numeric fields unscaled. Numeric magnitudes may influence SMOTENC neighbors, which remains a limitation of this fixed exploratory design.

In [ ]:
NOMINAL=CAT_NOT_APPLICABLE_COLS+CAT_UNKNOWN_RAW_COLS
CATEGORICAL=NOMINAL+ORDINAL_COLS
NUMERIC=NUM_LOG_CANDIDATES+NUM_OTHER+NUM_WITH_INDICATOR+SEMANTIC_FLAGS
assert len(CATEGORICAL)==18 and len(NUMERIC)==16 and len(CATEGORICAL+NUMERIC)==34
class MixedRepresentation(BaseEstimator,TransformerMixin):
 def fit(self,X,y=None):
  numeric=X[NUMERIC].apply(pd.to_numeric,errors="coerce")
  self.medians_=numeric.median().fillna(0)
  return self
 def transform(self,X):
  out=pd.DataFrame(index=X.index)
  for col in CAT_NOT_APPLICABLE_COLS:out[col]=X[col].fillna("Not applicable").astype(str)
  for col in CAT_UNKNOWN_RAW_COLS:out[col]=X[col].fillna("Unknown").astype(str)
  for col in ORDINAL_COLS:
   out[col]=X[col].fillna("Missing").astype(str)
   if not set(out[col]).issubset(ORDINAL_CATEGORY_ORDER):
    raise ValueError(f"Invalid committee category in {col}: {set(out[col])-set(ORDINAL_CATEGORY_ORDER)}")
  for col in NUMERIC:
   out[col]=pd.to_numeric(X[col],errors="coerce").fillna(self.medians_[col]).astype(float)
  assert list(out.columns)==CATEGORICAL+NUMERIC and not out.isna().any().any()
  assert np.isfinite(out[NUMERIC].to_numpy(dtype=float)).all()
  return out
class NamedSMOTENC(SMOTENC):
 def fit_resample(self,X,y):
  sampled,labels=super().fit_resample(X,y)
  if not isinstance(sampled,pd.DataFrame):sampled=pd.DataFrame(sampled,columns=X.columns)
  return sampled,labels
def to_float_array(values):
 return np.asarray(values,dtype=float)
def post_sampler_encoder():
 nominal_pipe=Pipeline([("onehot",OneHotEncoder(handle_unknown="infrequent_if_exist",
  min_frequency=LOCK["rare_min_frequency"],sparse_output=False))])
 ordinal_pipe=Pipeline([("ordinal",OrdinalEncoder(categories=[ORDINAL_CATEGORY_ORDER]*len(ORDINAL_COLS))),
  ("remap_missing",FunctionTransformer(remap_ordinal_missing_code,feature_names_out="one-to-one"))])
 numeric_pipe=Pipeline([("to_float",FunctionTransformer(to_float_array,feature_names_out="one-to-one"))])
 post=ColumnTransformer([("nominal",nominal_pipe,NOMINAL),
  ("ordinal",ordinal_pipe,ORDINAL_COLS),("numeric",numeric_pipe,NUMERIC)],
  remainder="drop",verbose_feature_names_out=True)
 post.set_output(transform="pandas")
 return post
def mixed_pipeline(sample):
 steps=[("fixed_cleaning",FixedDataCleaner()),("feature_engineering",FeatureEngineer()),
  ("mixed",MixedRepresentation())]
 if sample:steps.append(("sampler",NamedSMOTENC(categorical_features=list(range(len(CATEGORICAL))),
  random_state=42,sampling_strategy="auto",k_neighbors=5)))
 steps.extend([("post",post_sampler_encoder()),
  ("selection",SelectKBest(partial(mutual_info_classif,random_state=42),k=LOCK["feature_selection_k"])),
  ("classifier",RandomForestClassifier(random_state=42,n_jobs=2,**LOCK["hyperparameters"]))])
 return ImbPipeline(steps)
print("SMOTENC categorical columns:",len(CATEGORICAL),CATEGORICAL)
print("SMOTENC numeric columns:",len(NUMERIC),NUMERIC)

# 11. Structural SMOTENC safety gate
On one training fold, verify that the sampler sees only that fold, balances classes, preserves legal categorical values, and yields finite post-encoded numeric features with at least 60 columns. The untouched validation fold keeps its original row count. If any check fails, stop this experiment and report the technical reason; do not publish an invalid SMOTENC score.

In [ ]:
tr,va=OOF[0]
clean=FixedDataCleaner().fit_transform(X_train.iloc[tr])
fe=FeatureEngineer().fit(clean);engineered=fe.transform(clean)
mixed=MixedRepresentation().fit(engineered)
before=mixed.transform(engineered)
legal={col:set(before[col]) for col in CATEGORICAL}
sampler=NamedSMOTENC(categorical_features=list(range(len(CATEGORICAL))),
 random_state=42,sampling_strategy="auto",k_neighbors=5)
after,after_y=sampler.fit_resample(before,y.iloc[tr])
assert len(before)==len(tr) and len(after)>len(before)
assert len(va)==len(X_train.iloc[va]) and set(tr).isdisjoint(set(va))
assert list(after.columns)==CATEGORICAL+NUMERIC
assert all(set(after[col]).issubset(legal[col]) for col in CATEGORICAL)
assert not after.isna().any().any() and np.isfinite(after[NUMERIC].to_numpy(dtype=float)).all()
post=post_sampler_encoder()
encoded=post.fit_transform(after,after_y)
valid_mixed=mixed.transform(fe.transform(FixedDataCleaner().transform(X_train.iloc[va])))
valid_encoded=post.transform(valid_mixed)
assert encoded.shape[1]>=60 and valid_encoded.shape[0]==len(va)
assert np.isfinite(encoded.to_numpy(dtype=float)).all() and np.isfinite(valid_encoded.to_numpy(dtype=float)).all()
test_pipe=mixed_pipeline(True);test_pipe.fit(X_train.iloc[tr],y.iloc[tr])
test_valid=test_pipe.predict(X_train.iloc[va])
assert len(test_valid)==len(va)
smote_before_counts=pd.Series(y.iloc[tr]).value_counts().sort_index().to_dict()
smote_after_counts=pd.Series(after_y).value_counts().sort_index().to_dict()
print("SMOTENC before:",smote_before_counts,"after:",smote_after_counts)
print("SAFE SMOTENC: PASS; validation rows unchanged:",len(va))

# 12. SMOTENC control and sampled candidate
The control uses the same mixed representation and post-sampling encoder with the sampler omitted. This isolates the effect of sampling from the representation change. Both variants use the unchanged selector and locked RF parameters; each sampler is fit only inside its training fold.

In [ ]:
control_factory=lambda:mixed_pipeline(False)
smote_factory=lambda:mixed_pipeline(True)
control_folds=evaluate_repeated("Mixed representation control",control_factory)
control_folds.to_csv(RESULTS/"smotenc_control_folds.csv",index=False)
control_oof,control_pred,control_prob=evaluate_oof("Mixed representation control",control_factory)
smote_folds=evaluate_repeated("SMOTENC RF",smote_factory)
smote_folds.to_csv(RESULTS/"smotenc_folds.csv",index=False)
smote_oof,smote_pred,smote_prob=evaluate_oof("SMOTENC RF",smote_factory)
smote_summary=pd.DataFrame([summarize(control_folds),summarize(smote_folds)])
smote_summary.to_csv(RESULTS/"smotenc_summary.csv",index=False)
display(smote_summary.round(4))

# 13. Shared development comparison and class metrics
The repeated-fold table shows central performance and stability. OOF class metrics use exactly one validation prediction for each development row. Differences are descriptive and paired by split; they are not claims from a fresh external evaluation.

In [ ]:
all_folds=pd.concat([reference_folds,coordinate_folds,control_folds,smote_folds],ignore_index=True)
all_folds.to_csv(RESULTS/"posthoc_repeated_cv_folds.csv",index=False)
all_summary=pd.DataFrame([summarize(v) for v in [reference_folds,coordinate_folds,control_folds,smote_folds]])
all_summary.to_csv(RESULTS/"posthoc_repeated_cv_summary.csv",index=False)
oof_records=[]
for name,pred,prob in [("Locked RF",reference_pred,reference_prob),
 ("RF without coordinates",coordinate_pred,coordinate_prob),
 ("Mixed representation control",control_pred,control_prob),
 ("SMOTENC RF",smote_pred,smote_prob)]:
 p,r,f,s=precision_recall_fscore_support(y,pred,labels=CLASS_IDS,zero_division=0)
 for k,label in enumerate(CLASS_ORDER):
  oof_records.append({"configuration":name,"class":label,"support":int(s[k]),
   "precision":float(p[k]),"recall":float(r[k]),"f1":float(f[k]),
   "average_precision":float(average_precision_score((y==k).astype(int),prob[:,k]))})
oof_class=pd.DataFrame(oof_records)
oof_class.to_csv(RESULTS/"posthoc_oof_class_metrics.csv",index=False)
display(all_summary[["configuration","macro_f1","sd_macro_f1","mcc","partial_f1","partial_recall","train_cv_gap"]].round(4))

# 14. Country held-out development robustness
Leave one country out within development data. Each country is an unseen group at prediction time. The metric table records sample and class support, so small or missing-class countries are interpretable. The macro-F1 here averages the classes present in that country; it is compared within the same country across models, never pooled into the main CV score.

In [ ]:
groups=X_train["country"].fillna("Missing country").astype(str)
country_splits=list(LeaveOneGroupOut().split(X_train,y,groups))
assert len(country_splits)>=3
country_factories={"Locked RF":locked_pipeline,"RF without coordinates":coord_factory,
 "SMOTENC RF":smote_factory}
country_rows=[]
for name,factory in country_factories.items():
 for tr,va in country_splits:
  pipe=factory();pipe.fit(X_train.iloc[tr],y.iloc[tr]);pred=pipe.predict(X_train.iloc[va]).astype(int)
  country=groups.iloc[va].iloc[0];present=sorted(set(y.iloc[va]));support=y.iloc[va].value_counts().to_dict()
  country_rows.append({"configuration":name,"country":country,"n":len(va),
   "functional_n":support.get(0,0),"partial_n":support.get(1,0),"abandoned_n":support.get(2,0),
   "present_classes":len(present),"macro_f1_present":f1_score(y.iloc[va],pred,labels=present,average="macro",zero_division=0),
   "balanced_accuracy_present":balanced_accuracy_score(y.iloc[va],pred),
   "mcc":matthews_corrcoef(y.iloc[va],pred),"accuracy":accuracy_score(y.iloc[va],pred),
   "partial_recall":recall_score(y.iloc[va],pred,labels=[1],average="macro",zero_division=0) if 1 in present else np.nan})
country_df=pd.DataFrame(country_rows)
country_df.to_csv(RESULTS/"posthoc_country_robustness.csv",index=False)
display(country_df.pivot(index="country",columns="configuration",values="macro_f1_present").round(3))

# 15. Learning curves
Use five stratified folds and fixed train fractions. Compare the locked RF with the highest repeated-CV post-hoc candidate, using the same folds and Macro-F1. Curves diagnose whether the apparent gain comes with a larger training-validation gap. These are development diagnostics, not a model-selection search.

In [ ]:
candidates={"RF without coordinates":coord_factory,"Mixed representation control":control_factory,"SMOTENC RF":smote_factory}
strongest_name=max(candidates,key=lambda name:all_summary.set_index("configuration").loc[name,"macro_f1"])
curve_rows=[]
for name,factory in [("Locked RF",locked_pipeline),(strongest_name,candidates[strongest_name])]:
 sizes,train_scores,valid_scores=learning_curve(factory(),X_train,y,train_sizes=[.4,.6,.8,1.0],
  cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42),scoring="f1_macro",n_jobs=1,
  return_times=False)
 for i,size in enumerate(sizes):
  curve_rows.append({"configuration":name,"train_rows":int(size),
   "train_macro_f1_mean":float(train_scores[i].mean()),"train_macro_f1_sd":float(train_scores[i].std(ddof=1)),
   "cv_macro_f1_mean":float(valid_scores[i].mean()),"cv_macro_f1_sd":float(valid_scores[i].std(ddof=1)),
   "gap":float(train_scores[i].mean()-valid_scores[i].mean())})
curves=pd.DataFrame(curve_rows)
curves.to_csv(RESULTS/"learning_curve_diagnostics.csv",index=False)
display(curves.round(4))

# 16. Candidate assessment
A post-hoc candidate is marked promising only if mean repeated-CV Macro-F1 improves by more than 0.005, MCC does not fall by more than 0.005, Partial F1 and recall do not fall by more than 0.01, the mean train-CV gap does not widen by more than 0.02, and country-held-out mean Macro-F1 does not fall by more than 0.02. This predeclared, conservative screen is exploratory; it cannot change the official lock.

In [ ]:
summary_idx=all_summary.set_index("configuration")
ref=summary_idx.loc["Locked RF"]
country_means=country_df.groupby("configuration")["macro_f1_present"].mean().to_dict()
assessment=[]
for name in candidates:
 row=summary_idx.loc[name]
 checks={"macro_f1_gain_over_0_005":bool(row.macro_f1>ref.macro_f1+.005),
  "mcc_preserved":bool(row.mcc>=ref.mcc-.005),
  "partial_f1_preserved":bool(row.partial_f1>=ref.partial_f1-.01),
  "partial_recall_preserved":bool(row.partial_recall>=ref.partial_recall-.01),
  "train_cv_gap_controlled":bool(row.train_cv_gap<=ref.train_cv_gap+.02),
  "country_robustness_preserved":bool(country_means.get(name,-1)>=country_means["Locked RF"]-.02) if name in country_means else None,
  "sd_not_excessive":bool(row.sd_macro_f1<=ref.sd_macro_f1+.02)}
 eligible=all(v is True for v in checks.values())
 assessment.append({"configuration":name,"eligible":eligible,"checks":checks,
  "delta_macro_f1":float(row.macro_f1-ref.macro_f1),
  "delta_mcc":float(row.mcc-ref.mcc),
  "delta_partial_f1":float(row.partial_f1-ref.partial_f1),
  "delta_country_mean":float(country_means[name]-country_means["Locked RF"]) if name in country_means else None})
promising=[v for v in assessment if v["eligible"]]
chosen=max(promising,key=lambda v:v["delta_macro_f1"])["configuration"] if promising else None
recommendation={"official_locked_model_unchanged":True,"promising_posthoc_candidate":bool(chosen),"candidate_name":chosen,
 "screen":assessment,"conclusion":("Promising development-only candidate; formal reassessment required" if chosen else "Keep locked model")}
(RESULTS/"posthoc_candidate_assessment.json").write_text(json.dumps(recommendation,indent=2)+"\n",encoding="utf-8")
print("PROMISING POST-HOC CANDIDATE FOUND:","YES" if chosen else "NO")
display(pd.DataFrame([{k:v for k,v in item.items() if k!="checks"} for item in assessment]).round(4))

# 17. Figures and interpretation
Permutation importance reports validation-score reliance. The selected-feature figure reports RF impurity importance. Candidate comparisons display the primary metric and Partial-class recall. Learning curves show training and validation Macro-F1. Interpret correlated features and high-cardinality impurity importance cautiously.

In [ ]:
plt.rcParams.update({"figure.dpi":120,"axes.spines.top":False,"axes.spines.right":False})
fig,ax=plt.subplots(figsize=(8,6))
top=raw_importance.head(15).iloc[::-1]
ax.barh(top.raw_feature,top.mean_importance,xerr=top.sd_across_folds,color="#277da1",alpha=.85)
ax.axvline(0,color="black",lw=.8);ax.set_xlabel("Validation Macro-F1 decrease on permutation")
ax.set_title("Locked RF: raw feature permutation importance")
fig.tight_layout();fig.savefig(RESULTS/"raw_permutation_importance.png");plt.close(fig)
fig,ax=plt.subplots(figsize=(9,7))
top=processed_importance.head(20).iloc[::-1]
ax.barh(top.selected_processed_feature,top.rf_impurity_importance,color="#f8961e")
ax.set_xlabel("RF impurity importance");ax.set_title("Locked RF: selected processed features")
fig.tight_layout();fig.savefig(RESULTS/"selected_feature_importance.png");plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4))
view=all_summary.set_index("configuration")
axes[0].barh(view.index,view.macro_f1,xerr=view.sd_macro_f1,color="#43aa8b")
axes[0].set_xlabel("Repeated CV Macro-F1 (mean ± fold SD)")
axes[1].barh(view.index,view.partial_recall,color="#f3722c")
axes[1].set_xlabel("Repeated CV Partial recall")
fig.tight_layout();fig.savefig(RESULTS/"candidate_comparison.png");plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4))
for name,g in curves.groupby("configuration",sort=False):
 ax.plot(g.train_rows,g.train_macro_f1_mean,"--o",label=f"{name} train")
 ax.plot(g.train_rows,g.cv_macro_f1_mean,"-o",label=f"{name} CV")
ax.set(xlabel="Training rows per fold",ylabel="Macro-F1",title="Development learning curves")
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(RESULTS/"learning_curves.png");plt.close(fig)
print("Saved four figures in",RESULTS)

# 18. Main comparison and automated interpretation
The table and conclusions below are computed from development results. A smaller train–CV gap is useful only alongside validation performance. A country gain can identify a transfer-oriented candidate even when ordinary CV falls.

In [ ]:
comparison=all_summary.copy()
comparison["mean_country_macro_f1"]=comparison.configuration.map(country_means)
comparison=comparison[["configuration","macro_f1","sd_macro_f1","balanced_accuracy",
 "macro_recall","macro_precision","mcc","weighted_f1","accuracy","log_loss","macro_ap",
 "partial_precision","partial_recall","partial_f1","partial_ap","train_macro_f1",
 "train_cv_gap","mean_country_macro_f1"]]
comparison.to_csv(RESULTS/"posthoc_comparison.csv",index=False)
def delta(name,field):return float(summary_idx.loc[name,field]-ref[field])
ranked=lambda field:summary_idx[field].idxmax()
smallest_gap=summary_idx.train_cv_gap.idxmin()
raw_top=raw_importance.raw_feature.head(5).tolist()
processed_top=processed_importance.selected_processed_feature.head(5).tolist()
coord_country=float(country_means["RF without coordinates"]-country_means["Locked RF"])
smote_sample_delta=delta("SMOTENC RF","macro_f1")-delta("Mixed representation control","macro_f1")
robustness_candidate=(coord_country>=.03 and delta("RF without coordinates","macro_f1")>=-.02
 and delta("RF without coordinates","train_cv_gap")<0)
observations={
 "top_raw_permutation_features":raw_top,
 "top_selected_processed_features":processed_top,
 "coordinate_delta_cv_macro_f1":delta("RF without coordinates","macro_f1"),
 "coordinate_delta_train_cv_gap":delta("RF without coordinates","train_cv_gap"),
 "coordinate_delta_country_macro_f1":coord_country,
 "coordinate_delta_mcc":delta("RF without coordinates","mcc"),
 "coordinate_delta_partial_f1":delta("RF without coordinates","partial_f1"),
 "coordinate_delta_partial_recall":delta("RF without coordinates","partial_recall"),
 "smotenc_safe":True,
 "smotenc_delta_partial_recall_vs_control":float(summary_idx.loc["SMOTENC RF","partial_recall"]-summary_idx.loc["Mixed representation control","partial_recall"]),
 "smotenc_delta_partial_precision_vs_control":float(summary_idx.loc["SMOTENC RF","partial_precision"]-summary_idx.loc["Mixed representation control","partial_precision"]),
 "smotenc_delta_macro_f1_vs_control":smote_sample_delta,
 "smotenc_delta_mcc_vs_control":float(summary_idx.loc["SMOTENC RF","mcc"]-summary_idx.loc["Mixed representation control","mcc"]),
 "smotenc_delta_gap_vs_control":float(summary_idx.loc["SMOTENC RF","train_cv_gap"]-summary_idx.loc["Mixed representation control","train_cv_gap"]),
 "representation_control_delta_macro_f1_vs_locked":delta("Mixed representation control","macro_f1"),
 "best_cv_macro_f1":ranked("macro_f1"),"best_mcc":ranked("mcc"),
 "best_partial_f1":ranked("partial_f1"),"smallest_train_cv_gap":smallest_gap,
 "best_country_macro_f1":max(country_means,key=country_means.get),
 "coordinate_robustness_oriented_candidate":bool(robustness_candidate),
 "promising_overall_candidate":chosen}
(RESULTS/"automated_interpretation.json").write_text(json.dumps(observations,indent=2)+"\n",encoding="utf-8")
display(comparison.round(4))
for key,value in observations.items():print(f"{key}: {value}")
lines=["# Post-hoc development interpretation", "",
 f"Top raw validation permutation features: {', '.join(raw_top)}.",
 f"Top selected processed features by RF impurity importance: {', '.join(processed_top)}.",
 "Importance describes prediction reliance and tree split use, not causality.",
 f"Coordinate removal changed CV Macro-F1 by {delta('RF without coordinates','macro_f1'):+.4f}, MCC by {delta('RF without coordinates','mcc'):+.4f}, train-CV gap by {delta('RF without coordinates','train_cv_gap'):+.4f}, and mean country Macro-F1 by {coord_country:+.4f}.",
 f"SMOTENC versus its representation control changed Partial recall by {observations['smotenc_delta_partial_recall_vs_control']:+.4f}, Partial precision by {observations['smotenc_delta_partial_precision_vs_control']:+.4f}, Macro-F1 by {smote_sample_delta:+.4f}, MCC by {observations['smotenc_delta_mcc_vs_control']:+.4f}, and train-CV gap by {observations['smotenc_delta_gap_vs_control']:+.4f}.",
 f"The representation control changed Macro-F1 relative to the locked RF by {observations['representation_control_delta_macro_f1_vs_locked']:+.4f}; any SMOTENC-versus-locked difference includes that representation change.",
 f"Best CV Macro-F1: {ranked('macro_f1')}; best MCC: {ranked('mcc')}; best Partial F1: {ranked('partial_f1')}; smallest train-CV gap: {smallest_gap}; best mean country Macro-F1: {observations['best_country_macro_f1']}.",
 f"Coordinate-only ablation is a robustness-oriented candidate under the prespecified screen: {'yes' if robustness_candidate else 'no'}.",
 f"Promising overall post-hoc candidate: {chosen or 'none'}.",
 "The Notebook-05 RF remains the official Stage-7 model. Continue with it unless a separate formal reassessment and new external evaluation authorize a change."]
(RESULTS/"posthoc_interpretation.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("\n".join(lines))


# 19. Robustness dashboard
These panels compare the same repeated-CV folds and the mean across held-out countries. Fold standard deviations describe variability, not confidence intervals. Partial precision and recall must be read together.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,9))
view=comparison.set_index("configuration")
palette=["#277da1","#43aa8b","#f8961e","#f94144"]
for ax,field,title in zip(axes.flat,["macro_f1","mcc","partial_recall","partial_f1","partial_ap","mean_country_macro_f1"],
 ["CV Macro-F1","CV MCC","Partial recall","Partial F1","Partial AP","Mean country Macro-F1"]):
 ax.barh(view.index,view[field],color=palette)
 ax.set_title(title);ax.set_xlim(left=0)
fig.tight_layout();fig.savefig(RESULTS/"robustness_dashboard.png");plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(view));width=.36
ax.bar(x-width/2,view.train_macro_f1,width,label="Train",color="#277da1")
ax.bar(x+width/2,view.macro_f1,width,label="Validation",color="#f8961e")
ax.set_xticks(x,view.index,rotation=20,ha="right");ax.set_ylabel("Macro-F1")
ax.legend();fig.tight_layout();fig.savefig(RESULTS/"train_vs_cv_macro_f1.png");plt.close(fig)
print("Saved robustness dashboard and training versus CV figure")


# 20. Boundary and artifact audit
Verify the only model-data reads, original lock and Notebook-05 hashes, original sample count, and generated evidence. No code in this notebook reads the held-out features, labels, predictions or final metrics. The official model selection remains unchanged regardless of the post-hoc screen.

In [ ]:
assert set(Path(p).name for p in READ_PATHS)=={"X_train_raw.csv","y_train.csv"}
assert len(READ_PATHS)==2 and len(y)==1434
assert sha256(LOCK_PATH)==LOCK_HASH_BEFORE and sha256(NB05_PATH)==NB05_HASH_BEFORE
for forbidden in ["X_"+"test_raw.csv","y_"+"test.csv"]:
 assert all(forbidden not in p for p in READ_PATHS)
notebook_source=json.loads(Path("06b_posthoc_robustness.ipynb").read_text(encoding="utf-8"))
code_source="".join("".join(cell["source"]) for cell in notebook_source["cells"] if cell["cell_type"]=="code")
for forbidden in ["X_"+"test_raw.csv","y_"+"test.csv","final_"+"holdout_metrics.json"]:
 assert forbidden not in code_source
assert (RESULTS/"posthoc_candidate_assessment.json").exists()
audit={"development_rows":len(y),"model_data_reads":READ_PATHS,
 "lock_sha256_before_after":LOCK_HASH_BEFORE,"notebook05_sha256_before_after":NB05_HASH_BEFORE,
 "official_model_unchanged":True,"holdout_access":"NONE",
 "evidence_files":sorted(p.name for p in RESULTS.iterdir() if p.is_file())}
(RESULTS/"posthoc_audit.json").write_text(json.dumps(audit,indent=2)+"\n",encoding="utf-8")
print("POST-HOC HOLDOUT ACCESS: NONE")
print("OFFICIAL LOCK UNCHANGED: YES")